# Week 7 Exercise — Fine-tune for "The Price is Right" (profe-ssor)

Fine-tune an **open-source model** (e.g. Llama, Qwen) to predict a product's **price** from its description.

- **Data:** Course items (`items_lite` / `items_full`) via `pricer.items.Item.from_hub`.
- **Training:** QLoRA (see week7 day3/day4 notebooks); prepare prompts with `item.make_prompts(tokenizer, CUTOFF, do_round)`.
- **Eval:** Use `pricer.evaluator.evaluate(predictor, test)` to score your model.

Run this notebook from `week7/` or from this folder; the first code cell adds `week7` to the path so `pricer` can be imported.

In [ ]:
import os
import sys
from pathlib import Path

# Run from week7/ or from week7/community_contributions/profe-ssor/
cwd = Path.cwd()
if cwd.name == "profe-ssor":
    week7 = cwd.parent.parent
else:
    week7 = cwd
if str(week7) not in sys.path:
    sys.path.insert(0, str(week7))

from dotenv import load_dotenv
load_dotenv(override=True)

## Config

Set `LITE_MODE = True` to use the small dataset; use your HuggingFace username for the dataset.

In [ ]:
LITE_MODE = True
username = "ed-donner"  # or your HF username if you pushed your own dataset
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

In [ ]:
from huggingface_hub import login
import os
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)

## Load data and tokenizer

Load items from the Hub and the base model tokenizer (for prompt formatting and, later, inference).

In [ ]:
from pricer.items import Item
from transformers import AutoTokenizer

train, val, test = Item.from_hub(dataset)
print(f"Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}")

BASE_MODEL = "meta-llama/Llama-3.2-3B"  # or e.g. Qwen/Qwen2.5-3B-Instruct
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

## Prepare prompts

Build prompt/completion for each item (same format as week7 day2). Use a `CUTOFF` to truncate long summaries.

In [ ]:
from tqdm.notebook import tqdm

CUTOFF = 110
for item in tqdm(train + val, desc="Train/Val prompts"):
    item.make_prompts(tokenizer, CUTOFF, do_round=True)
for item in tqdm(test, desc="Test prompts"):
    item.make_prompts(tokenizer, CUTOFF, do_round=False)

print("Example prompt (test):")
print(test[0].prompt)
print("Completion:", test[0].completion)

## (Optional) Train with QLoRA

Use the week7 day3/day4 notebooks to run QLoRA training (e.g. on Colab). Save your adapter or merged model to the Hub, then load it below for evaluation.

In [ ]:
# After training, load your model and tokenizer, e.g.:
# from peft import PeftModel
# from transformers import AutoModelForCausalLM
# base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, ...)
# model = PeftModel.from_pretrained(base, "your-repo/your-adapter")
# Then implement predict_price_from_item using model.generate(...) and parse the number.

## Predictor for evaluation

`pricer.evaluator.evaluate` expects a **function(item) -> price**. Use `item.test_prompt()` to get the prompt without the answer, then run your model and parse the price from the completion.

In [ ]:
def predict_price_from_item(item):
    """Predict price for one Item. Replace with your fine-tuned model inference."""
    # Placeholder: use true price (no model). Replace with:
    # prompt = item.test_prompt()
    # completion = your_model.generate(prompt, ...)
    # return float(parse_price_from_completion(completion))
    return item.price

## Evaluate

Run the course evaluator on the test set (subset size and workers are configurable).

In [ ]:
from pricer.evaluator import evaluate

evaluate(predict_price_from_item, test, size=200, workers=4)